# Importación y exportación de archivos

## Importación de librerías y variables de entorno

### Librerías de trabajo

In [ ]:
# I/O tabular y transformaciones
import pandas as pd

# Operaciones numéricas
import numpy as np

# Manejo robusto de rutas multiplataforma
from pathlib import Path

# constantes de 'quoting' para CSV
import csv

### Variables de entorno

In [ ]:
# Carpeta de trabajo actual
BASE_DIR = Path().resolve().parent.parent

# Carpeta donde están las bases de datos
DATA_DIR = BASE_DIR / 'data'

# Archivo de entrada esperado (UCI)
IN_FILE  = DATA_DIR / 'cirrosis' / 'cirrhosis.csv'

# Carpeta destino para exportaciones
OUT_DIR  = DATA_DIR / 'out'

# Crea la carpeta si no existe
OUT_DIR.mkdir(parents=True, exist_ok=True)

## CSV: formato universal y de texto

### Lectura con opciones frecuentes

In [ ]:
df = pd.read_csv(
    # Ruta al archivo
    IN_FILE,
    # Delimitador
    sep=',',
    # Codificación. Alternativas: 'latin-1', 'cp1252', etc
    encoding='utf-8',
    # Literales a tratar como datos faltantes
    na_values=['NA', ''],
    # Inferir tipo de datos columnas. Es posible pasar diccionarios, por ejemplo {'Status':'category', 'N_Days':'Int64'}
    dtype={'Status':'category'},
    # Puede ser True o lista de columnas de fecha
    parse_dates=False,
    # False si las fechas tienen el formato MM/DD/YYYY
    dayfirst=True,
    # Pueden pasar una lista de las columnas a seleccionar, por ejemplo ['ID', 'Status','Bilirubin']
    usecols=None,
    # Número de filas a leer
    nrows=None,
    # Número de líneas a saltar antes de leer
    skiprows=None,
    # Evita tipos mixtos de datos
    low_memory=False,
    # Separador decimal
    decimal='.',
    # Separador de miles
    thousands=None,
    # Detecta por extensión, puede ser None, 'gzip', 'bz2', 'zip', 'xz' o dict
    compression='infer'
)

Ver las dimensiones del dataframe.

In [ ]:
df.shape

Ver el tipo de dato de cada variable.

In [ ]:
df.dtypes

### Escritura + compresión y formato

Ejemplo con compresión gzip

In [ ]:
df.to_csv(
    # Ruta donde guardar el archivo
    OUT_DIR / 'cirrosis.csv.gz',
    # Escribir índice como columna
    index=False,
    # Separador
    sep=',',
    # Codificación
    encoding='utf-8',
    # Cómo represantar datos faltantes
    na_rep='',
    # Formato de flotantes
    float_format='%.4f',
    # Manejo de comillas: MINIMAL|ALL|NONNUMERIC|NONE
    quoting=csv.QUOTE_MINIMAL,
    # Tipo de compresión None|'gzip'|'bz2'|'xz'|'zip'|dict
    compression='gzip'
)

## Excel: reporte y multihoja

### Lectura

OJO: Requiere tener instalada la librería openpyxl

In [ ]:
df_xlsx = pd.read_excel(
    # Ruta una vez que ya se tiene el archivo
    DATA_DIR / 'cirrosis.xlsx',
    # Hoja por índice o por nombre. Puede ser lista para seleccionar varias.
    sheet_name=0,
    # Columnas a leer, por ejemplo 'A:C,F'
    usecols=None,
    # Diccionario de tipos de datos de las columnas
    dtype=None,
    # Literales a tratar como datos faltantes
    na_values=['NA'],
    # Motor recomendado para lectura
    engine='openpyxl'
)

### Escritura

Los archivos con extensión .xlsx ya están comprimidos internamente por lo que el tamaño suele ser razonable. Para formateo avanzado el motor 'xlswriter' ofrece más opciones como ancho de columnas, congelar paneles, entre otros.

In [ ]:
# Ruta donde guardar el archivo
xlsx_out = OUT_DIR / 'cirrosis.xlsx'
with pd.ExcelWriter(xlsx_out, engine='openpyxl') as writer:
    # Primera hoja con toda la base
    df.to_excel(writer, sheet_name='Base', index=False)

In [ ]:
# Ruta donde guardar el archivo
xlsx_out = OUT_DIR / 'reporte_cirrosis.xlsx'
with pd.ExcelWriter(xlsx_out, engine='openpyxl') as writer:
    # Primera hoja con toda la base
    df.to_excel(writer, sheet_name='Base', index=False)
    # Segunda hoja con estadísticas descriptivas
    df.describe().to_excel(writer, sheet_name='Resumen', index=True)

## JSON: datos semi estructurados no tabulares

### Lectura

JSON 'clásico'

In [ ]:
df_json  = pd.read_json(
    # Ruta una vez que se tiene el archivo
    OUT_DIR / 'cirrosis.json',
    # Tipo de estructura
    orient='records'
)

Según el consumo, el parámetro orient puede ser:
* *records*: APIs / web (lista de diccionarios)
* *table*: incluye **schema**
* *split | index | columns | values*: estructuras compactas para pandas

JSON Lines

In [ ]:
df_jsonl = pd.read_json(
    # Ruta donde se encuentra el archivo
    OUT_DIR / 'cirrosis.ndjson.gz',
    # Tipo de escritura
    orient='records',
    # Una fila por línea
    lines=True,
    # Método de compresión
    compression='gzip'
)

### Escritura

JSON 'clásico'

In [ ]:
df.to_json(
    # Ruta donde guardar el archivo
    OUT_DIR / 'cirrosis.json',
    # Se tiene lista de objetos {col: valor}
    orient='records',
    # Sangría para legibilidad
    indent=2,
    # Permite caracteres que no son ascii
    force_ascii=False,
    # Formato ISO para fechas
    date_format='iso',
    # Decimales para floats
    double_precision=10
)

JSON Lines

In [ ]:
df.to_json(
    # Ruta donde guardar el archivo
    OUT_DIR / 'cirrosis.ndjson.gz',
    # Cada fila es un objeto JSON
    orient='records',
    # Una fila por línea
    lines=True,
    # Método de compresión
    compression='gzip'
)

## Parquet

### Lectura

In [ ]:
df_parquet = pd.read_parquet(
    # Ruta donde se encuentra el archivo
    OUT_DIR / 'cirrosis.parquet',
    # Motor, puede ser pyarrow o fastparquet
    engine='pyarrow'
)

### Escritura

In [ ]:
df.to_parquet(
    # Ruta donde guardar el archivo
    OUT_DIR / 'cirrosis.parquet',
    # Motor
    engine='pyarrow',
    # Guardar o no el índice como columna
    index=False,
    # Método de compresión, puede ser snappy, gzip, brotli, zstd entre otros
    compression='snappy'
)

## Feather: intercambio veloz entre Python y R

### Lectura

In [ ]:
df_feather = pd.read_feather(
    # Ruta donde se ecuentra el archivo
    OUT_DIR / 'cirrosis.feather'
)

### Escritura

In [ ]:
df.to_feather(
    # Ruta donde se quiere guardar la base
    OUT_DIR / 'cirrosis.feather',
    # Método de compresión, puede ser 'lz4' para mejor velocidad o 'zstd' para mayor compresión
    compression='lz4'
)

## Pickle

### Lectura

In [ ]:
df_from_pickle = pd.read_pickle(
    # Ruta donde se encuentra la base
    OUT_DIR / 'cirrosis_df.pkl'
)

### Escritura

In [ ]:
df.to_pickle(
    # Ruta donde se quiere guardar la base
    OUT_DIR / 'cirrosis_df.pkl',
    # Protocolo -1 usa el más alto disponible
    protocol=-1
)


## Pipelines o modelos con joblib

Supongamos que tenemos un pipeline 'preprocess' de preprocesamiento o un modelo 'clf' ya entrenado.

### Lectura

In [ ]:
preprocess2 = joblib.load(
    # Ruta donde se encuentra guardado el objeto
    OUT_DIR / 'preprocess.joblib'
)

### Escritura

In [ ]:
joblib.dump(
    # Objeto a guardar
    preprocess,
    # Ruta donde guardarlo
    OUT_DIR / 'preprocess.joblib',
    # Calidad de compresión, de 0 (sin compresión) a 9 (máxima)
    # Puede ser entero o tuple (zlib | gzip | bz2 | xz, nivel)
    compress=3
)

## Compresión

Además de los parámetros *compression* de pandas se pueden crear archivos comprimidos con la librería estándar.

### Seleccionando archivos

In [ ]:
from zipfile import ZipFile, ZIP_DEFLATED
import shutil

# Rutas donde hay archivos guardados
rutas = [
    # Archivo csv
    DATA_DIR / 'cirrosis' / 'cirrhosis.csv',
    # Archivo xlsx
    OUT_DIR / 'cirrosis.xlsx',
    # Archuvo parquet
    OUT_DIR / 'cirrosis.parquet',
    # Archivo Feather
    OUT_DIR / 'cirrosis.feather',
    # Archivo JSON clásico
    OUT_DIR / 'cirrosis.json',
    # Archivo JSON Lines
    OUT_DIR / 'cirrosis.ndjson.gz'
]
with ZipFile(
    # Ruta donde guardar el archivo
    OUT_DIR / 'export_bundle.zip',
    # Modo (lectura, escritura, binario...)
    mode="w",
    # Método de compresión
    compression=ZIP_DEFLATED
) as zf:
    # Para cada ruta de tipo de archivo
    for p in rutas:
        # Verifica si existe el archivo
        if p.exists():
            zf.write(
                # Archivo
                p,
                # Nombre dentro del ZIP
                arcname=p.name
            )

### Directamente el directorio

In [ ]:
# Ruta donde escribir el directorio comprimido
archive_base = OUT_DIR.with_name('io_demo_archive')

# Crea el folder io_demo_archive.zip
shutil.make_archive(
    # Nombre del directorio
    str(archive_base), 
    # Método de compresión, puede ser zip, gzip, bz2 o xz
    "zip",
    # Directorio en donde guardar el comprimido
    root_dir= OUT_DIR
)  

## Lecturas por chunks

In [ ]:
status_counts = {}
for chunk in pd.read_csv(IN_FILE, chunksize=50_000, na_values=['NA'], encoding='utf-8'):
    counts = chunk['Status'].value_counts(dropna=False).to_dict()
    for k, v in counts.items():
        status_counts[k] = status_counts.get(k, 0) + v
    print('Conteos de Status (acumulado):', status_counts)

## Validación de round-trip

Guardar -> Leer -> Comprobar

In [ ]:
tmp_pq = OUT_DIR / 'roundtrip.parquet'
df.to_parquet(tmp_pq, engine='pyarrow', index=False, compression='snappy')
df_rt = pd.read_parquet(tmp_pq)

Comprobar mismas columnas en mismo orden

In [ ]:
same_cols = (list(df.columns) == list(df_rt.columns))

Comprobar igualdad estructural

In [ ]:
same_len  = (len(df) == len(df_rt))

Resultados

In [ ]:
print("Mismas columnas:", same_cols, " | Mismo número de filas:", same_len)